In [95]:
import tensorflow as tf
import pandas as pd
import numpy as np

from scipy.optimize import minimize_scalar, minimize

In [96]:
def gradient(f, x):
    x_tensor = tf.Variable(np.array(x, dtype=float), dtype=tf.float32)
    
    with tf.GradientTape() as tape:
        tape.watch(x_tensor)
        y = f(x_tensor)
    
    grad = tape.gradient(y, x_tensor).numpy()
    return grad

# Метод Гаусса-Зейделя

In [97]:
def gauss_seidel(f, x0, epsilon=1e-6, max_iter=1000):
    x_k = np.array(x0, dtype=float)
    n = len(x_k)
    
    for _ in range(max_iter):
        y_j = x_k.copy()
        
        for j in range(n):
            def phi(lambda_j):
                y_temp = y_j.copy()
                y_temp[j] += lambda_j
                return f(y_temp)
            
            lambda_j = minimize_scalar(phi).x
            
            y_j[j] = y_j[j] + lambda_j

        if np.linalg.norm(y_j - x_k) < epsilon:
            return y_j, f(y_j)
        
        x_k = y_j.copy()

    print("Максимальное число итераций достигнуто.")
    return x_k, f(x_k)

# Метод наискорейшего спуска

In [98]:
def steepest_descent(f, x0, epsilon=1e-6, max_iter=1000):
    x_k = np.array(x0, dtype=float)

    for _ in range(max_iter):
        grad_x_k = gradient(f, x_k)

        direction = -grad_x_k

        phi = lambda lambda_k: f(x_k + lambda_k * direction)
        res = minimize_scalar(phi)
        lambda_k = res.x

        x_next = x_k + lambda_k * direction

        if np.linalg.norm(x_next - x_k) < epsilon:
            return x_next, f(x_next)
        
        x_k = x_next
        
    print("Максимальное число итераций достигнуто.")
    return x_k, f(x_k)

# Метод Хука и Дживса

In [99]:
def hooke_jeeves(f, x0, epsilon=1e-6, delta=1e-2, max_iter=1000):  
    x_k = np.array(x0, dtype=float)
    n = len(x_k)
    
    for _ in range(max_iter):
        x_new = x_k.copy()

        for i in range(n):
            x_temp = x_k.copy()
            x_temp[i] += delta

            if f(x_temp) < f(x_new):
                x_new = x_temp

            x_temp = x_k.copy()
            x_temp[i] -= delta

            if f(x_temp) < f(x_new):
                x_new = x_temp
        
        if np.linalg.norm(x_new - x_k) < epsilon:
            return x_new, f(x_new)
        
        if f(x_new) < f(x_k):
            x_k = x_new
        else:
            delta *= 0.5
    
    print("Максимальное число итераций достигнуто.")
    return x_k, f(x_k)

# Метод Розенброка

In [100]:
def rosenbrock(f, x0, epsilon=1e-6, max_iter=1000):
    x_k = np.array(x0, dtype=float)

    for _ in range(max_iter):
        grad_x_k = gradient(f, x_k)

        direction = grad_x_k / np.linalg.norm(grad_x_k)

        result = minimize(lambda lambda_: f(x_k + lambda_ * direction), 0)
        lambda_k = result.x[0]

        x_next = x_k + lambda_k * direction

        if np.linalg.norm(x_next - x_k) < epsilon:
            return x_next, f(x_next)

        x_k = x_next

    print("Максимальное число итераций достигнуто.")
    return x_k, f(x_k)

# Проверка алгоритмов

In [101]:
f = lambda x: 20 - (x[0] - 1) * np.e ** (- (x[0] - 1)) - (x[1] - 2) * np.e ** (- (x[1] - 2))
x0 = [0.0, 0.0]

methods = [("Решение из библиотеки", lambda: minimize(f, x0)),
           ("Гаусс-Зейдель", lambda: gauss_seidel(f, x0)),
           ("Наискорейший спуск", lambda: steepest_descent(f, x0)),
           ("Хука и Дживса", lambda: hooke_jeeves(f, x0)),
           ("Розенброка", lambda: rosenbrock(f, x0))]

results = []

for method_name, method_func in methods:
    result = method_func()
    if isinstance(result, tuple):
        point, value = result
        results.append([method_name, [f"{x:.4f}" for x in point], f"{value:.4f}"])
    else:
        results.append([method_name, [f"{x:.4f}" for x in result.x], f"{result.fun:.4f}"])

df = pd.DataFrame(results, columns=["Метод", "X*", "F(X*)"])
df


,Метод,X*,F(X*)
0,Решение из библиотеки,"[2.0000, 3.0000]",19.2642
1,Гаусс-Зейдель,"[2.0000, 3.0000]",19.2642
2,Наискорейший спуск,"[2.0000, 3.0000]",19.2642
3,Хука и Дживса,"[2.0000, 3.0000]",19.2642
4,Розенброка,"[2.0000, 3.0000]",19.2642
